# Bài tập: Dự đoán **Churn** khách hàng viễn thông (Telco) bằng Cây quyết định & Rừng ngẫu nhiên

**Mục tiêu:** Xây pipeline tiền xử lý, huấn luyện **Decision Tree** (DecisionTreeClassifier) & **Random Forest** (RandomForestClassifier), đánh giá bằng **Stratified K-Fold CV** & **hold-out**, tuning, **Permutation Importance**, **lược đặc trưng**.
**Tài liệu tham khảo**:https://www.w3schools.com/python/python_ml_decision_tree.asp
https://www.w3schools.com/python/python_ml_cross_validation.asp
https://www.w3schools.com/python/python_ml_auc_roc.asp

**Dữ liệu:** Kaggle — *Telco Customer Churn: https://www.kaggle.com/datasets/graceegbe12/advanced-customer-churn-prediction*. Đặt CSV vào `./telco-data/`.

## 0) Import & Chuẩn bị

In [1]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path('./telco-data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

## 1) EDA

In [ ]:
# TODO: đọc file, chuyển TotalCharges -> numeric, xem phân bố nhãn
candidates = ['WA_Fn-UseC_-Telco-Customer-Churn.csv', 'Telco-Customer-Churn.csv']
csv_path = None
for name in candidates:
    fp = DATA_DIR / name
    if fp.exists():
        csv_path = fp
        break
assert csv_path is not None, f"Hãy đặt file Telco CSV vào {DATA_DIR.resolve()} (ví dụ: {candidates[0]})"
df = pd.read_csv(csv_path)
df['TotalCharges'] = pd.to_numeric(df.get('TotalCharges', pd.Series(dtype=float)), errors='coerce')
display(df.head())

## 2) Chuẩn bị X, y và nhóm cột

In [ ]:
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn', 'customerID'], errors='ignore')
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
len(num_cols), len(cat_cols)

## 3) Pipeline tiền xử lý

In [ ]:
numeric_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
# OneHotEncoder tương thích phiên bản
ohe_kwargs = dict(handle_unknown='ignore')
try:
    ohe = OneHotEncoder(sparse_output=False, **ohe_kwargs)  # sklearn >= 1.2
except TypeError:
    ohe = OneHotEncoder(sparse=False, **ohe_kwargs)         # sklearn < 1.2

categorical_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', ohe)])
preprocess = ColumnTransformer([('num', numeric_pipe, num_cols), ('cat', categorical_pipe, cat_cols)])
preprocess.fit(X)
feat_names = preprocess.get_feature_names_out()
len(feat_names)

## 4) Baseline: Tree & RF (Stratified 5-fold CV)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc':'roc_auc', 'accuracy':'accuracy', 'f1':'f1'}
# TODO: tạo pipeline tree & rf, cross_validate và báo cáo

## 5) Hold-out 80/20 và đánh giá

In [ ]:
# TODO: stratified split, fit, dự đoán, tính AUC/ACC/F1, in report & confusion matrix

## 6) Tuning (Grid cho Tree, Randomized cho RF, scoring='roc_auc')

In [ ]:
# TODO: GridSearchCV/RandomizedSearchCV, in best params & best score

## 7) Permutation Importance (validation)

In [ ]:
# TODO: permutation_importance(scoring='roc_auc'), in Top-20 và vẽ biểu đồ

## 8) Lược đặc trưng Top-K & tái huấn luyện

In [ ]:
# TODO: chọn Top-K theo importance, tạo selector, CV lại với RF

## 9) (Tuỳ chọn) OOB & tối ưu ngưỡng

In [ ]:
# TODO: OOB score; dò threshold tối ưu theo F1